# 00｜pandas 與資料視覺化：用股市資料學工具

**本章定位：** 工具熱身——學完這章，你就有足夠的能力跑完後面所有的分析

## 先說一件事

你可能已經學過 pandas。會 `read_csv()`、會 `groupby()`、會篩選欄位。

但有一個問題：**你用這些工具，問過什麼有意思的問題嗎？**

這章不打算從頭教 pandas 語法。
我們直接拿真實的股市資料，用你已經知道（或快要知道）的工具，問幾個值得問的問題。

- 台積電過去十年漲了多少？
- 跟 0050 比，誰的波動比較大？
- 每天的漲跌幅，長什麼樣子？
- 移動平均線到底在看什麼？

這些問題，用 20 行 Python 就能回答。

## 🎯 學習目標
1. 用 `yfinance` 下載真實股市資料，熟悉 pandas DataFrame 的基本操作
2. 計算日報酬率與累積報酬，理解「報酬」的兩種表達方式
3. 用 matplotlib 畫出股價走勢圖、報酬分布圖、移動平均圖

---

## 一｜下載股市資料

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

try:
    import yfinance as yf
    HAS_YFINANCE = True
except ImportError:
    HAS_YFINANCE = False
    print('請先安裝 yfinance：pip install yfinance')

# 中文字型設定
plt.rcParams['font.family'] = ['Arial Unicode MS', 'Heiti TC', 'sans-serif']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

In [ ]:
# 下載 0050 和台積電 2015 年至今的收盤價
TICKERS = ['0050.TW', '2330.TW']
NAMES   = {'0050.TW': '0050', '2330.TW': '台積電'}
START   = '2015-01-01'

if HAS_YFINANCE:
    raw = yf.download(TICKERS, start=START, auto_adjust=True, progress=False)
    prices = raw['Close'].copy()
    # yfinance 版本差異：有時回傳 MultiIndex
    if isinstance(prices.columns, pd.MultiIndex):
        prices.columns = prices.columns.get_level_values(0)
    prices = prices.dropna()
else:
    print('無法下載資料，請確認 yfinance 已安裝並有網路連線')

print(f'資料筆數：{len(prices)} 個交易日')
print(f'時間範圍：{prices.index[0].date()} ～ {prices.index[-1].date()}')
prices.tail(3)

### 快速認識資料

`DataFrame.describe()` 可以一次看到基本統計量——最大值、最小值、平均、標準差。
對股市資料來說，**標準差**特別重要，它代表波動程度。

In [ ]:
prices.describe().round(1)

---

## 二｜計算報酬率

股價本身不太好比較——台積電股價 600 元，0050 股價 150 元，哪個「漲比較多」？

所以我們用**報酬率**：每一天相對於前一天，漲了或跌了幾個百分比。

$$r_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$

pandas 的 `.pct_change()` 幫你直接算好。

In [ ]:
# 日報酬率
returns = prices.pct_change().dropna()

print('日報酬率統計（%）：')
print((returns * 100).describe().round(3))

In [ ]:
# 累積報酬：把每天的 (1 + 報酬率) 連乘
# 例如：第一天漲 2%，第二天漲 3%，累積 = 1.02 × 1.03 - 1 = 5.06%
cum_returns = (1 + returns).cumprod() - 1

# 2015 年至今，各自漲了多少？
print('\n從 2015 年至今的累積報酬：')
for col in cum_returns.columns:
    name = NAMES.get(col, col)
    total = cum_returns[col].iloc[-1] * 100
    print(f'  {name:5s}：{total:>8.1f}%')

---

## 三｜視覺化

數字看完，來畫圖。matplotlib 的基本結構就是：

```python
fig, ax = plt.subplots()   # 建立畫布
ax.plot(x, y)              # 畫線
ax.set_title('標題')        # 加標題
plt.show()                 # 顯示
```

多張子圖就用 `plt.subplots(列數, 欄數)`。

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# 上圖：股價走勢
for col in prices.columns:
    axes[0].plot(prices.index, prices[col], label=NAMES.get(col, col), linewidth=1.2)
axes[0].set_title('股價走勢（2015 年至今）')
axes[0].set_ylabel('股價（元）')
axes[0].legend()

# 下圖：累積報酬
for col in cum_returns.columns:
    axes[1].plot(cum_returns.index, cum_returns[col] * 100,
                 label=NAMES.get(col, col), linewidth=1.2)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('累積報酬率（%）')
axes[1].set_ylabel('累積報酬（%）')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 報酬率分布：每天漲跌幅長什麼樣子？
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for i, col in enumerate(returns.columns):
    name = NAMES.get(col, col)
    r = returns[col] * 100
    avg = r.mean()
    std = r.std()

    axes[i].hist(r, bins=60, color='steelblue', edgecolor='none', alpha=0.75)
    axes[i].axvline(avg, color='red', linestyle='--', linewidth=1.5,
                    label=f'平均 {avg:.2f}%')
    axes[i].axvline(avg + std, color='orange', linestyle=':', linewidth=1.2,
                    label=f'+1σ = {avg+std:.2f}%')
    axes[i].axvline(avg - std, color='orange', linestyle=':', linewidth=1.2,
                    label=f'-1σ = {avg-std:.2f}%')
    axes[i].set_title(f'{name} 日報酬分布')
    axes[i].set_xlabel('日報酬率（%）')
    axes[i].legend(fontsize=9)

plt.suptitle('報酬率分布：大多數日子漲跌幅在哪個範圍？', y=1.01)
plt.tight_layout()
plt.show()

print('\n波動度比較（日報酬標準差）：')
for col in returns.columns:
    name = NAMES.get(col, col)
    print(f'  {name}：±{returns[col].std()*100:.2f}% / 天')

---

## 四｜移動平均線

移動平均線是最常見的技術指標之一——把過去 N 天的收盤價平均起來，畫成一條線。

它的作用不是「預測未來」，而是**過濾短期雜訊，看出中期趨勢**。

pandas 的 `.rolling(N).mean()` 就搞定了。

> 等等的實證課程會說明：移動平均線的「訊號」，在統計上到底有沒有預測力。

In [ ]:
p0050 = prices['0050.TW']

ma20  = p0050.rolling(20).mean()   # 月線（約 20 個交易日）
ma60  = p0050.rolling(60).mean()   # 季線（約 60 個交易日）
ma240 = p0050.rolling(240).mean()  # 年線（約 240 個交易日）

fig, ax = plt.subplots(figsize=(13, 5))

ax.plot(p0050.index, p0050,   label='0050 收盤價', color='#4C72B0', linewidth=1,   alpha=0.8)
ax.plot(ma20.index,  ma20,    label='月線 (MA20)',  color='#DD8452', linewidth=1.5)
ax.plot(ma60.index,  ma60,    label='季線 (MA60)',  color='#55A868', linewidth=1.5)
ax.plot(ma240.index, ma240,   label='年線 (MA240)', color='#C44E52', linewidth=2)

ax.set_title('0050 收盤價與移動平均線')
ax.set_ylabel('股價（元）')
ax.legend()

plt.tight_layout()
plt.show()

---

## 五｜比較兩資產的關係

最後一個問題：0050 和台積電的漲跌，有多像？

用**相關係數**量化：1 表示完全同向，0 表示毫無關係，-1 表示反向。

In [ ]:
corr = returns.corr()
print('日報酬相關係數矩陣：')
print(corr.round(3))

r_val = corr.loc['0050.TW', '2330.TW']
print(f'\n0050 與台積電的相關係數：{r_val:.3f}')
print(f'→ 台積電佔 0050 權重約 30%+，所以兩者高度相關')

In [ ]:
# 散布圖：每個點是「某一天 0050 的報酬 vs 台積電的報酬」
fig, ax = plt.subplots(figsize=(6, 6))

ax.scatter(returns['0050.TW'] * 100,
           returns['2330.TW'] * 100,
           alpha=0.15, s=10, color='steelblue')

# 加上 45 度參考線
lim = max(abs(returns.values.min()), abs(returns.values.max())) * 100 * 1.05
ax.plot([-lim, lim], [-lim, lim], 'r--', linewidth=1, alpha=0.5, label='y = x')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)

ax.set_xlabel('0050 日報酬（%）')
ax.set_ylabel('台積電 日報酬（%）')
ax.set_title(f'0050 vs 台積電日報酬散布（相關係數 = {r_val:.3f}）')
ax.legend()

plt.tight_layout()
plt.show()

---

## 這跟你有什麼關係？

今天這幾個操作——下載資料、算報酬率、畫圖、看相關係數——就是後面課程所有分析的地基。

但工具只是工具。有了工具之後，更重要的問題是：

- 移動平均線穿越是不是真的有預測力？（→ Notebook 08 月份效應會討論）
- 台積電和 0050 這麼相關，持有兩個算不算分散風險？（→ Notebook 10、11 因子投資）
- 過去十年報酬這麼好，未來還會一樣嗎？（→ Notebook 03 CAPE 估值）

這些問題，光靠畫圖是回答不了的，需要統計檢定和學術文獻的支撐。

那就是後面的課程要做的事。

---

## 📌 本章重點摘要

| 工具 | 用途 | 關鍵語法 |
|------|------|----------|
| `yfinance` | 下載股市資料 | `yf.download(ticker, start=...)` |
| `pct_change()` | 計算日報酬率 | `df.pct_change().dropna()` |
| `cumprod()` | 計算累積報酬 | `(1 + r).cumprod() - 1` |
| `rolling().mean()` | 移動平均線 | `df.rolling(20).mean()` |
| `corr()` | 相關係數矩陣 | `df.corr()` |
| `plt.subplots()` | 多子圖畫布 | `fig, axes = plt.subplots(2, 1)` |
| `ax.hist()` | 報酬分布直方圖 | `ax.hist(data, bins=60)` |

> **下一章：** 行為金融學——你的大腦在投資上為什麼這麼不可靠？